# Jetstream2: new instance — one-time setup & training commands

Use this notebook as a **checklist**. Run commands on the **remote VM** in a terminal (SSH), not necessarily inside Jupyter unless you start a notebook server on the instance.

**Assumptions**
- You created a Linux VM (e.g. Ubuntu 22.04) on [Jetstream2](https://jetstream-cloud.org/) with a **floating IP** and your **SSH public key** in `~/.ssh/authorized_keys`.
- Default cloud user is often `ubuntu` or `exouser` — replace `USER` and `INSTANCE_IP` below.

---

## 1. From your laptop: SSH into the instance

Replace `INSTANCE_IP` with the VM’s public IP (or hostname) and `USER` with the login user Jetstream gave you.

In [ ]:
# Run locally (Mac/Linux). Windows: use PowerShell or WSL similarly.
ssh USER@INSTANCE_IP

If you use a specific key file:

```bash
ssh -i ~/.ssh/your_jetstream_key.pem USER@INSTANCE_IP
```

**First time:** accept the host key when prompted (`yes`).

---

## 2. On the VM: system packages (once per fresh image)

Ubuntu/Debian example:

In [ ]:
sudo apt update && sudo apt upgrade -y
sudo apt install -y git python3 python3-venv python3-pip build-essential tmux htop rsync

---

## 3. Project directory and Python venv

Adjust `PROJECT` if you prefer another path.

**Always activate the venv** before `python src/train.py`: `source .venv/bin/activate`. If you see `ModuleNotFoundError: No module named 'torch'`, the venv is missing packages — run `pip install -r requirements.txt` again (includes **PyTorch**). For a **GPU** build, after the CPU install works you can reinstall PyTorch using the selector at [pytorch.org](https://pytorch.org/get-started/locally/).

In [ ]:
mkdir -p ~/projects && cd ~/projects

# Option A: clone from GitHub (HTTPS — needs token/credentials, or SSH — see step 4)
# git clone https://github.com/MahsaAbadian/brain_cta.git TopBrain_Algo_Submission
# cd TopBrain_Algo_Submission

# Option B: rsync from your laptop (see step 5) — then:
# cd ~/projects/TopBrain_Algo_Submission

python3 -m venv .venv
source .venv/bin/activate
pip install --upgrade pip
pip install -r requirements.txt

---

## 4. (Optional) GitHub over SSH from the VM

Useful if the repo is private or you want to `git pull` without HTTPS tokens.

**On the VM:**

In [ ]:
ssh-keygen -t ed25519 -C "jetstream-vm" -f ~/.ssh/id_ed25519 -N ""
cat ~/.ssh/id_ed25519.pub

Copy the **public** key line and add it in GitHub: **Settings → SSH and GPG keys → New SSH key**.

Test:

In [ ]:
ssh -T git@github.com

Expected: `Hi USERNAME! You've successfully authenticated...`

Then clone with SSH:

```bash
cd ~/projects
git clone git@github.com:MahsaAbadian/brain_cta.git TopBrain_Algo_Submission
cd TopBrain_Algo_Submission
python3 -m venv .venv && source .venv/bin/activate && pip install -r requirements.txt
```

---

## 5. (Optional) Copy the **code repo** from your laptop with `rsync`

Run **on your laptop** (not on the VM). Replace paths, user, and IP.

This syncs the **whole project folder** (code + whatever data you have locally). For **only** `training_data/` or `training_data_resampled/`, see **section 6** instead.

In [ ]:
# Run on LOCAL machine — example:
 rsync -avz --progress \
  -e "ssh -i ~/.ssh/id_ed25519" \
  /Users/mahsaabadian/MachineLearning/Nazim/TopBrain_Algo_Submission \
   exouser@149.165.151.119:~/topbrain/

After rsync, on the VM:

```bash
cd ~/projects/TopBrain_Algo_Submission
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

---

## 6. Get the TopBrain dataset (download or copy from your machine)

### Where to download

- **Zenodo (TopBrain 2025 release):** [https://zenodo.org/records/16878417](https://zenodo.org/records/16878417) — download the archive(s) listed there (large; use a stable connection).
- **Challenge / documentation:** [https://topbrain2025.grand-challenge.org](https://topbrain2025.grand-challenge.org) — data page and citation details.
- This repo only ships `training_data/README.txt` and `training_data/License.txt`; **scans and labels are not in git**.

### Layout after you unpack (raw training data)

Under the **project root** (same layout as `README.md` **Put the Raw Data Here** in this repo):

- `training_data/imagesTr_topbrain_ct/*.nii.gz`
- `training_data/labelsTr_topbrain_ct/*.nii.gz`
- (optional MR) `training_data/imagesTr_topbrain_mr/`, `training_data/labelsTr_topbrain_mr/`
- `training_data/itksnap_labelmap_txt/` (label maps — keep with the data you use)

### Option A: Download directly on the VM

1. `cd ~/projects/TopBrain_Algo_Submission` (or your project path).
2. Use **wget/curl** with the Zenodo file link(s) from your browser (right‑click the download button → copy URL), or download in the browser on your laptop and use **Option B**.
3. Unzip so the folders above sit under `training_data/`.

### Option B: Copy from your laptop with `rsync` (recommended if you already have the data locally)

Run **on your laptop** (replace paths, user, IP, and optional `-e` SSH key). Sync **into** the project directory on the VM.

**Raw data only** (then preprocess on the VM — step 7):

```bash
rsync -avz --progress \
  -e "ssh -i ~/.ssh/your_key" \
  /path/to/local/TopBrain_Algo_Submission/training_data/ \
  USER@INSTANCE_IP:~/projects/TopBrain_Algo_Submission/training_data/
```

**Preprocessed data** (skip long preprocessing on the VM if your laptop already has `training_data_resampled/`):

```bash
rsync -avz --progress \
  -e "ssh -i ~/.ssh/id_ed25519" \
  /Users/mahsaabadian/MachineLearning/Nazim/TopBrain_Algo_Submission \
   exouser@149.165.151.119:~/topbrain/
```

Smaller alternative: **`scp -r`** the same folders (slower than rsync for large datasets).

---

## 7. Data layout the training script expects

Training defaults use **resampled** CTA paths (see `src/train.py` / `src/data_loader.py`):

- `training_data_resampled/imagesTr_topbrain_ct/`
- `training_data_resampled/labelsTr_topbrain_ct/`
- `training_data_resampled/split/` (`train_cases.txt`, `val_cases.txt`)
- `training_data_resampled/itksnap_labelmap_txt/labelmap_topbrain_ct.txt`

If you copied **only raw** `training_data/` to the VM, run **once** from the project root (after `source .venv/bin/activate`):

```bash
python src/preprocess_resample.py --dataset topbrain_ct --copy-metadata
```

That writes into `training_data_resampled/`. If `split/` or labelmaps are missing, copy them from your laptop or re-run split generation per `DOCUMENTATION.md`.

---

## 8. Long runs: `tmux` (recommended)

SSH disconnects will not kill the training job if it runs inside a tmux session.

**On the VM:**

In [ ]:
tmux new -s train

Inside tmux: activate venv, `cd` to project, run training (section 9).

**Detach** (leave job running): `Ctrl+b` then `d`  
**Reattach:** `tmux attach -t train`  
**List sessions:** `tmux ls`

---

## 9. Training command (GPU if available)

From project root with venv activated. Adjust `--out-dir`, epochs, and patch settings to your GPU memory.

In [ ]:
cd ~/projects/TopBrain_Algo_Submission
source .venv/bin/activate

python src/train.py \
  --epochs 200 \
  --lr 1e-3 \
  --num-patches-per-volume 8 \
  --num-val-patches-per-volume 8 \
  --patch-size 96 96 96 \
  --out-dir runs/cluster_$(date +%Y%m%d)

Checkpoints and `metrics.csv` appear under `--out-dir`.

**CUDA:** If `nvidia-smi` works, PyTorch should use GPU automatically (`device=cuda` in the script log).

---

## 10. Copy results back to your laptop (optional)

**On your laptop:**

In [ ]:
# rsync -avz USER@INSTANCE_IP:~/projects/TopBrain_Algo_Submission/runs/ \
#   ./runs_from_jetstream/

---

## 11. Quick sanity checks on the VM

```bash
cd ~/projects/TopBrain_Algo_Submission
source .venv/bin/activate
python -c "import torch; print('cuda:', torch.cuda.is_available())"
pytest -q
```

---

## Notes

- **Security groups / firewall:** OpenStack security groups must allow **SSH (port 22)** from your IP to connect.
- **Floating IP:** If the instance is deleted, the IP may change; update your SSH config and `rsync` targets.
- **Jupyter on the VM:** If you want JupyterLab in the browser, install it in the venv and bind to `0.0.0.0` with a password; **only** do this on a network you trust and lock down the security group to your IP.

For training details, see `TRAINING.md` in the repo root.